In [1]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Load UCI promoter dataset directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"

df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

In [2]:
def clean_sequence(seq):
    seq = seq.upper()
    seq = seq.strip()

    return seq
df_cleaned = df.copy()
df_cleaned['label'] = df_cleaned['label'].map({'-': 0, '+': 1})
df_cleaned['sequence'] = df_cleaned['sequence'].apply(clean_sequence)

In [3]:
class PromoterDataset(Dataset):
    def __init__(self, sequences, labels, encoder):
        self.sequences = sequences
        self.labels = labels
        self.encoder = encoder
        self.encoded_sequences = [self.encoder.encode(seq).T for seq in self.sequences]
        self.encoded_labels = torch.tensor(self.labels, dtype=torch.float32)
        self.encoded_sequences = torch.stack(self.encoded_sequences) # Transpose to (4,57)
    
    def __len__(self):
        return len(self.encoded_sequences)
    
    def __getitem__(self, idx):
        return self.encoded_sequences[idx], self.encoded_labels[idx]

In [49]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import sys
sys.path.append('../src/')
from UpdatedSequenceEncoder import SequenceEncoder
from LSTMClassifier import LSTMClassifier

# Prepare data
sequences = df_cleaned['sequence'].values
labels = df_cleaned['label'].values

#Dataset
dataset = PromoterDataset(sequences, labels, SequenceEncoder())

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
fold_losses = []
encoder = SequenceEncoder()

for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels)):
    print(f"\n{'='*40}")
    print(f"Fold {fold+1}/5")
    print(f"  Train: {len(train_idx)} samples")
    print(f"  Test:  {len(test_idx)} samples")
    
    # 1. Split data using train_idx, test_idx
    train_sequences, train_labels = dataset.encoded_sequences[train_idx], dataset.encoded_labels[train_idx]
    test_sequences, test_labels = dataset.encoded_sequences[test_idx], dataset.encoded_labels[test_idx]
    
    # 2. Create FRESH model + optimizer + criterion

    model = LSTMClassifier(input_size=4, hidden_size=57)
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters())
    # 4. Train for 50 epochs
    for epoch in range(50):
        model.train()
        train_loss = 0
        for X_train, y_train in zip(train_sequences, train_labels):
            y_train = y_train.float().unsqueeze(-1)  # Ensure labels are float for BCELoss
            optimizer.zero_grad()
            outputs = model(X_train.T)  # Transpose to (57,4) for LSTM input
            loss = criterion(outputs, y_train)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
    #Fold-level train loss (optional)
    print(f"Train Loss: {train_loss/len(train_sequences):.4f}")

    # 5. Evaluate on test fold
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        final_loss = 0
        for X_test, y_test in zip(test_sequences, test_labels):
            predictions = model(X_test.T)
            predicted_labels = (predictions > 0.5).float()
            correct += (predicted_labels.squeeze() == y_test).sum().item()
            total += 1
            final_loss += criterion(predictions, y_test.float().unsqueeze(-1)).item()
    print(f"  Final test loss:  {final_loss/len(test_labels):.4f}")
    # 7. Append accuracy to fold_accuracies
    fold_losses.append(final_loss / len(test_labels))  # Average loss per sample  
    accuracy = correct / total if total > 0 else 0
    fold_accuracies.append(accuracy)
# If train loss << test loss → overfitting
# If train loss ≈ test loss → good generalization

# Results
print(f"\n{'='*40}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*40}")
print(f"Fold accuracies: {[f'{a:.2%}' for a in fold_accuracies]}")
print(f"Mean accuracy:   {np.mean(fold_accuracies):.2%}")
print(f"Std deviation:   {np.std(fold_accuracies):.2%}")


Fold 1/5
  Train: 84 samples
  Test:  22 samples
Train Loss: 0.6972
  Final test loss:  0.7013

Fold 2/5
  Train: 85 samples
  Test:  21 samples
Train Loss: 0.1203
  Final test loss:  1.0051

Fold 3/5
  Train: 85 samples
  Test:  21 samples
Train Loss: 0.2950
  Final test loss:  0.8964

Fold 4/5
  Train: 85 samples
  Test:  21 samples
Train Loss: 0.3785
  Final test loss:  0.7343

Fold 5/5
  Train: 85 samples
  Test:  21 samples
Train Loss: 0.6523
  Final test loss:  0.7505

CROSS-VALIDATION RESULTS
Fold accuracies: ['54.55%', '57.14%', '61.90%', '57.14%', '38.10%']
Mean accuracy:   53.77%
Std deviation:   8.19%


In [48]:
X_test.shape, len(test_sequences)

(torch.Size([4, 57]), 22)

In [33]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import sys

from sympy import sequence
sys.path.append('../src/')
from UpdatedSequenceEncoder import SequenceEncoder
from LSTMTrainer import Model

# Prepare data
sequences = df_cleaned['sequence'].values
labels = df_cleaned['label'].values

#Dataset
dataset = PromoterDataset(sequences, labels, SequenceEncoder())

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
fold_losses = []
encoder = SequenceEncoder()

for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels)):
    print(f"\n{'='*40}")
    print(f"Fold {fold+1}/5")
    print(f"  Train: {len(train_idx)} samples")
    print(f"  Test:  {len(test_idx)} samples")
    
    # 1. Split data using train_idx, test_idx
    train_sequences, train_labels = dataset.encoded_sequences[train_idx], dataset.encoded_labels[train_idx]
    test_sequences, test_labels = dataset.encoded_sequences[test_idx], dataset.encoded_labels[test_idx]
    
    # 2. Create FRESH model + optimizer + criterion

    model = Model(input_size=4, hidden_size=57)
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters())
    # 4. Train for 50 epochs
    for epoch in range(50):
        model.train()
        train_loss = 0
        for X_train, y_train in zip(train_sequences, train_labels):
            c_prev = torch.zeros(57)
            h_prev = torch.zeros(57)
            y_train = y_train.float().unsqueeze(-1)  # Ensure labels are float for BCELoss
            optimizer.zero_grad()
            _, _, output = model(X_train.T, c_prev, h_prev)  # Transpose to (57,4) for LSTM input
            loss = criterion(output, y_train)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
    #Fold-level train loss (optional)
    print(f"  Train Loss: {train_loss/len(train_sequences):.4f}")

    # 5. Evaluate on test fold
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        final_loss = 0
        for X_test, y_test in zip(test_sequences, test_labels):
            _, _, predictions = model(X_test.T, torch.zeros(57), torch.zeros(57))
            predicted_labels = (predictions > 0.5).float()
            correct += (predicted_labels.squeeze() == y_test).sum().item()
            total += y_test.sum().item()
            final_loss += criterion(predictions, y_test.float().unsqueeze(-1)).item()
    print(f"  Final test loss:  {final_loss/len(test_labels):.4f}")
    # 7. Append accuracy to fold_accuracies
    fold_losses.append(final_loss / len(test_labels))  # Average loss per sample  
    accuracy = correct / total if total > 0 else 0
    fold_accuracies.append(accuracy)
# If train loss << test loss → overfitting
# If train loss ≈ test loss → good generalization

# Results
print(f"\n{'='*40}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*40}")
print(f"Fold accuracies: {[f'{a:.2%}' for a in fold_accuracies]}")
print(f"Mean accuracy:   {np.mean(fold_accuracies):.2%}")
print(f"Std deviation:   {np.std(fold_accuracies):.2%}")


Fold 1/5
  Train: 84 samples
  Test:  22 samples
  Train Loss: 0.1253
  Final test loss:  1.5429

Fold 2/5
  Train: 85 samples
  Test:  21 samples
  Train Loss: 0.4577
  Final test loss:  0.7561

Fold 3/5
  Train: 85 samples
  Test:  21 samples
  Train Loss: 0.0556
  Final test loss:  1.4099

Fold 4/5
  Train: 85 samples
  Test:  21 samples
  Train Loss: 0.3710
  Final test loss:  0.8214

Fold 5/5
  Train: 85 samples
  Test:  21 samples
  Train Loss: 0.2340
  Final test loss:  1.7043

CROSS-VALIDATION RESULTS
Fold accuracies: ['127.27%', '127.27%', '109.09%', '100.00%', '100.00%']
Mean accuracy:   112.73%
Std deviation:   12.33%


In [32]:
final_loss/len(test_labels)

0.8938608302601746